# 03b_Feature Engineering

## Objective
Build the canonical feature-engineering notebook for neighbour-aware drift modelling.

This notebook takes validated neighbour relationships and hourly sensor data, then creates a clean feature table for downstream fault scoring.

This notebook does not perform final fault scoring or final fault type classification.

---

## Inputs
- `/data/merged_sensor_metadata.csv`
- `/data/hourly_sensor_panel.csv`
- `/data/metric_neighbour_coherence.csv`
- `/data/validated_neighbour_pairs.csv`

If any missing, stop with diagnostic.

---

## Assumptions
- Time column: `timestamp_utc`
- Sensor id: `ateccid`
- All relevant numeric metrics modeled together
- Decision horizon: 24 hours
- Sensors with no neighbours: self-history only
- Feature naming: clean, stable, collision-safe

Avoid feature-name duplication bugs.

---

## Outputs
1. `sensor_hour_features.csv`
2. `sensor_window_features.csv`
3. `feature_dictionary.csv`
4. `sensor_neighbour_metadata.csv`
5. `model_metric_list.csv`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import json


%matplotlib inline

sns.set(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', 200)

root = Path('..')
data_dir = root / 'data'
outputs = data_dir

required_files = {
    'merged': data_dir / 'merged_sensor_metadata.csv',
    'hourly_panel': data_dir / 'hourly_sensor_panel.csv',
    'coherence': data_dir / 'metric_neighbour_coherence.csv',
    'pairs': data_dir / 'validated_neighbour_pairs.csv'
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {name} at {path.resolve()}")

print('All prerequisite files found.')

All prerequisite files found.


In [2]:
# 1. Load prerequisite artifacts
hourly_panel = pd.read_csv(required_files['hourly_panel'])
coherence_df = pd.read_csv(required_files['coherence'])
pairs_df = pd.read_csv(required_files['pairs'])

# Load merged for metadata if needed
merged = pd.read_csv(required_files['merged'])

timestamp_col = 'timestamp_utc'
sensor_col = 'ateccid'

# Ensure types
hourly_panel[timestamp_col] = pd.to_datetime(hourly_panel[timestamp_col])
hourly_panel[sensor_col] = hourly_panel[sensor_col].astype(str)

# Get metrics from coherence
all_metrics = coherence_df['metric'].unique().tolist()
included_metrics = coherence_df.loc[coherence_df['usable_for_neighbour_monitoring'].isin(['yes', 'review']), 'metric'].tolist()
excluded_metrics = coherence_df.loc[coherence_df['usable_for_neighbour_monitoring'] == 'no', 'metric'].tolist()
review_metrics = coherence_df.loc[coherence_df['usable_for_neighbour_monitoring'] == 'review', 'metric'].tolist()

print('All metrics:', all_metrics)
print('Included metrics:', included_metrics)
print('Excluded metrics:', excluded_metrics)
print('Review metrics:', review_metrics)

# For this notebook, use included + review
model_metrics = included_metrics + review_metrics
print('Model metrics:', model_metrics)

# Save model metric list
model_metric_list_path = outputs / 'model_metric_list.csv'
pd.DataFrame({'metric': model_metrics}).to_csv(model_metric_list_path, index=False)
print('Saved model metric list:', model_metric_list_path)

All metrics: ['capacity_people', 'min_occupant', 'max_occupant', 'total_occupant', 'tvoc', 'co2', 'temp', 'pressure', 'lux', 'snd', 'combined_occupancy', 'humid', 'aqi1', 'aqi2', 'aqi3', 'blue_relative', 'clear_relative', 'green_relative', 'no2', 'o3', 'red_relative', 'voc_resistance', 'avg_occupant', 'occupancy1', 'occupancy2']
Included metrics: ['tvoc', 'co2', 'temp', 'pressure', 'lux', 'snd', 'humid', 'aqi1', 'aqi2', 'blue_relative', 'clear_relative', 'green_relative', 'red_relative', 'voc_resistance']
Excluded metrics: ['capacity_people', 'min_occupant', 'max_occupant', 'total_occupant', 'combined_occupancy', 'aqi3', 'no2', 'o3', 'avg_occupant', 'occupancy1', 'occupancy2']
Review metrics: []
Model metrics: ['tvoc', 'co2', 'temp', 'pressure', 'lux', 'snd', 'humid', 'aqi1', 'aqi2', 'blue_relative', 'clear_relative', 'green_relative', 'red_relative', 'voc_resistance']
Saved model metric list: ../data/model_metric_list.csv


In [3]:
# 2. Build neighbour-aware hourly features
# Prepare wide tables per metric
wide_tables = {}
for metric in model_metrics:
    wide_tables[metric] = (
        hourly_panel
        .pivot(index=timestamp_col, columns=sensor_col, values=metric)
        .sort_index()
    )

# Get sensor list and neighbour map
sensor_list = sorted(hourly_panel[sensor_col].unique())
neighbour_map = pairs_df.groupby('sensor')['neighbour'].apply(list).to_dict()

# Function to compute neighbour stats
def compute_neighbour_stats(sensor, metric, ts, wide_df, neigh_map):
    neighs = neigh_map.get(sensor, [])
    if not neighs:
        return {
            'neighbour_mean': np.nan,
            'neighbour_median': np.nan,
            'neighbour_std': np.nan,
            'neighbour_mad': np.nan,
            'neighbour_count': 0,
            'neighbour_coverage_ratio': 0.0
        }
    
    neigh_values = wide_df.loc[ts, neighs].dropna()
    count = len(neigh_values)
    ratio = count / len(neighs) if neighs else 0.0
    
    if count == 0:
        return {
            'neighbour_mean': np.nan,
            'neighbour_median': np.nan,
            'neighbour_std': np.nan,
            'neighbour_mad': np.nan,
            'neighbour_count': count,
            'neighbour_coverage_ratio': ratio
        }
    
    mean_val = neigh_values.mean()
    median_val = neigh_values.median()
    std_val = neigh_values.std()
    mad_val = (neigh_values - median_val).abs().median()
    
    return {
        'neighbour_mean': mean_val,
        'neighbour_median': median_val,
        'neighbour_std': std_val,
        'neighbour_mad': mad_val,
        'neighbour_count': count,
        'neighbour_coverage_ratio': ratio
    }

# Build feature table
feature_records = []
for idx, row in hourly_panel.iterrows():
    sensor = row[sensor_col]
    ts = row[timestamp_col]
    
    record = {
        timestamp_col: ts,
        sensor_col: sensor,
        'neighbour_mode': 'neighbour' if neighbour_map.get(sensor) else 'self',
        'neighbour_count': len(neighbour_map.get(sensor, [])),
        'effective_neighbour_count': 0,  # will fill
        'fallback_reason': None
    }
    
    # Compute sensor-level coverage (average across first metric or use first available)
    sensor_coverage_ratio = 0.0
    
    for metric in model_metrics:
        raw_val = row[metric]
        record[f'{metric}_raw'] = raw_val
        
        if pd.isna(raw_val):
            # Skip if no data
            continue
        
        neigh_stats = compute_neighbour_stats(sensor, metric, ts, wide_tables[metric], neighbour_map)
        record.update({f'{metric}_{k}': v for k, v in neigh_stats.items()})
        
        # Capture sensor-level coverage from first metric
        if sensor_coverage_ratio == 0.0 and not pd.isna(neigh_stats['neighbour_coverage_ratio']):
            sensor_coverage_ratio = neigh_stats['neighbour_coverage_ratio']
        
        # Residuals
        if not pd.isna(neigh_stats['neighbour_mean']):
            record[f'{metric}_residual_mean'] = raw_val - neigh_stats['neighbour_mean']
        if not pd.isna(neigh_stats['neighbour_median']):
            record[f'{metric}_residual_median'] = raw_val - neigh_stats['neighbour_median']
            record[f'{metric}_abs_residual'] = abs(record[f'{metric}_residual_median'])
            if neigh_stats['neighbour_mad'] > 0:
                record[f'{metric}_robust_z'] = record[f'{metric}_residual_median'] / neigh_stats['neighbour_mad']
        
        # Relative residual if sensible
        if not pd.isna(neigh_stats['neighbour_median']) and neigh_stats['neighbour_median'] != 0:
            record[f'{metric}_rel_residual'] = record[f'{metric}_residual_median'] / abs(neigh_stats['neighbour_median'])
    
    # Add sensor-level coverage ratio
    record['neighbour_coverage_ratio'] = sensor_coverage_ratio
    
    feature_records.append(record)

hour_features = pd.DataFrame(feature_records)
print('Hour features shape:', hour_features.shape)
print('Sample hour features:')
hour_features.head()

Hour features shape: (25344, 175)
Sample hour features:


,timestamp_utc,ateccid,neighbour_mode,neighbour_count,effective_neighbour_count,fallback_reason,tvoc_raw,tvoc_neighbour_mean,tvoc_neighbour_median,tvoc_neighbour_std,tvoc_neighbour_mad,tvoc_neighbour_count,tvoc_neighbour_coverage_ratio,co2_raw,co2_neighbour_mean,co2_neighbour_median,co2_neighbour_std,co2_neighbour_mad,co2_neighbour_count,co2_neighbour_coverage_ratio,temp_raw,temp_neighbour_mean,temp_neighbour_median,temp_neighbour_std,temp_neighbour_mad,temp_neighbour_count,temp_neighbour_coverage_ratio,pressure_raw,pressure_neighbour_mean,pressure_neighbour_median,pressure_neighbour_std,pressure_neighbour_mad,pressure_neighbour_count,pressure_neighbour_coverage_ratio,lux_raw,lux_neighbour_mean,lux_neighbour_median,lux_neighbour_std,lux_neighbour_mad,lux_neighbour_count,lux_neighbour_coverage_ratio,snd_raw,snd_neighbour_mean,snd_neighbour_median,snd_neighbour_std,snd_neighbour_mad,snd_neighbour_count,snd_neighbour_coverage_ratio,humid_raw,humid_neighbour_mean,humid_neighbour_median,humid_neighbour_std,humid_neighbour_mad,humid_neighbour_count,humid_neighbour_coverage_ratio,aqi1_raw,aqi1_neighbour_mean,aqi1_neighbour_median,aqi1_neighbour_std,aqi1_neighbour_mad,aqi1_neighbour_count,aqi1_neighbour_coverage_ratio,aqi2_raw,aqi2_neighbour_mean,aqi2_neighbour_median,aqi2_neighbour_std,aqi2_neighbour_mad,aqi2_neighbour_count,aqi2_neighbour_coverage_ratio,blue_relative_raw,blue_relative_neighbour_mean,blue_relative_neighbour_median,blue_relative_neighbour_std,blue_relative_neighbour_mad,blue_relative_neighbour_count,blue_relative_neighbour_coverage_ratio,clear_relative_raw,clear_relative_neighbour_mean,clear_relative_neighbour_median,clear_relative_neighbour_std,clear_relative_neighbour_mad,clear_relative_neighbour_count,clear_relative_neighbour_coverage_ratio,green_relative_raw,green_relative_neighbour_mean,green_relative_neighbour_median,green_relative_neighbour_std,green_relative_neighbour_mad,green_relative_neighbour_count,green_relative_neighbour_coverage_ratio,red_relative_raw,red_relative_neighbour_mean,red_relative_neighbour_median,red_relative_neighbour_std,red_relative_neighbour_mad,red_relative_neighbour_count,red_relative_neighbour_coverage_ratio,voc_resistance_raw,voc_resistance_neighbour_mean,voc_resistance_neighbour_median,voc_resistance_neighbour_std,voc_resistance_neighbour_mad,voc_resistance_neighbour_count,voc_resistance_neighbour_coverage_ratio,neighbour_coverage_ratio,tvoc_residual_mean,tvoc_residual_median,tvoc_abs_residual,tvoc_robust_z,tvoc_rel_residual,co2_residual_mean,co2_residual_median,co2_abs_residual,co2_robust_z,co2_rel_residual,temp_residual_mean,temp_residual_median,temp_abs_residual,temp_robust_z,temp_rel_residual,pressure_residual_mean,pressure_residual_median,pressure_abs_residual,pressure_rel_residual,lux_residual_mean,lux_residual_median,lux_abs_residual,lux_robust_z,lux_rel_residual,snd_residual_mean,snd_residual_median,snd_abs_residual,snd_robust_z,snd_rel_residual,humid_residual_mean,humid_residual_median,humid_abs_residual,humid_robust_z,humid_rel_residual,aqi1_residual_mean,aqi1_residual_median,aqi1_abs_residual,aqi1_robust_z,aqi1_rel_residual,aqi2_residual_mean,aqi2_residual_median,aqi2_abs_residual,aqi2_robust_z,aqi2_rel_residual,blue_relative_residual_mean,blue_relative_residual_median,blue_relative_abs_residual,blue_relative_robust_z,blue_relative_rel_residual,clear_relative_residual_mean,clear_relative_residual_median,clear_relative_abs_residual,clear_relative_robust_z,clear_relative_rel_residual,green_relative_residual_mean,green_relative_residual_median,green_relative_abs_residual,green_relative_robust_z,green_relative_rel_residual,red_relative_residual_mean,red_relative_residual_median,red_relative_abs_residual,voc_resistance_residual_mean,voc_resistance_residual_median,voc_resistance_abs_residual,voc_resistance_robust_z,voc_resistance_rel_residual,pressure_robust_z,red_relative_robust_z,red_relative_rel_residual
0,2025-08-15 00:00:00+00:00,0123067D065E709F01,self,0,0,None,0.0,NaN,Na

In [4]:
# 3. Add self-history fallback features
# For sensors with no neighbours or low coverage, add self-history features
def add_self_history_features(df, sensor_col, timestamp_col, metrics, window_hours=[6, 12, 24]):
    df = df.sort_values([sensor_col, timestamp_col]).copy()
    df = df.set_index(timestamp_col)
    
    for metric in metrics:
        raw_col = f'{metric}_raw'
        if raw_col not in df.columns:
            continue
        
        for wh in window_hours:
            # Rolling stats with time-based window
            df[f'{metric}_self_mean_{wh}h'] = (
                df.groupby(sensor_col)[raw_col]
                .rolling(window=f'{wh}h', min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
            df[f'{metric}_self_median_{wh}h'] = (
                df.groupby(sensor_col)[raw_col]
                .rolling(window=f'{wh}h', min_periods=1)
                .median()
                .reset_index(level=0, drop=True)
            )
            df[f'{metric}_self_std_{wh}h'] = (
                df.groupby(sensor_col)[raw_col]
                .rolling(window=f'{wh}h', min_periods=1)
                .std()
                .reset_index(level=0, drop=True)
            )
            df[f'{metric}_self_mad_{wh}h'] = (
                df.groupby(sensor_col)[raw_col]
                .apply(lambda x: (x - x.rolling(window=f'{wh}h', min_periods=1).median()).abs().rolling(window=f'{wh}h', min_periods=1).median())
                .reset_index(level=0, drop=True)
            )
    
    # Deviation from self-baseline
    for metric in metrics:
        if f'{metric}_self_median_24h' in df.columns and f'{metric}_raw' in df.columns:
            df[f'{metric}_self_deviation'] = df[f'{metric}_raw'] - df[f'{metric}_self_median_24h']
    
    # Rolling slope (simple linear trend)
    for metric in metrics:
        raw_col = f'{metric}_raw'
        if raw_col in df.columns:
            df[f'{metric}_self_slope_24h'] = (
                df.groupby(sensor_col)[raw_col]
                .rolling(window=24, min_periods=6)
                .apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) >= 6 else np.nan)
                .reset_index(level=0, drop=True)
            )
    
    # Instability score (rolling std of residuals to median)
    for metric in metrics:
        if f'{metric}_self_median_24h' in df.columns and f'{metric}_raw' in df.columns:
            resid = df[f'{metric}_raw'] - df[f'{metric}_self_median_24h']
            df[f'{metric}_self_instability'] = (
                resid.groupby(df[sensor_col])
                .rolling(window=24, min_periods=6)
                .std()
                .reset_index(level=0, drop=True)
            )
    
    df = df.reset_index()
    return df

# Apply fallback features
hour_features = add_self_history_features(hour_features, sensor_col, timestamp_col, model_metrics)

# Update fallback reason
hour_features['fallback_reason'] = np.where(
    hour_features['neighbour_mode'] == 'self',
    'no_neighbours',
    np.where(
        hour_features['neighbour_coverage_ratio'] < 0.5,
        'low_coverage',
        None
    )
)

print('After self-history features:', hour_features.shape)

/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/1365671857.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{metric}_self_median_{wh}h'] = (
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/1365671857.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{metric}_self_std_{wh}h'] = (
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/1365671857.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, wh

After self-history features: (25344, 385)


/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/1365671857.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{metric}_self_instability'] = (
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/1365671857.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.reset_index()


In [5]:
# 4. Add temporal features
hour_features['hour_of_day'] = hour_features[timestamp_col].dt.hour
hour_features['day_of_week'] = hour_features[timestamp_col].dt.dayofweek
hour_features['weekend_flag'] = hour_features['day_of_week'].isin([5, 6]).astype(int)

# Cyclical encodings
hour_features['hour_sin'] = np.sin(2 * np.pi * hour_features['hour_of_day'] / 24)
hour_features['hour_cos'] = np.cos(2 * np.pi * hour_features['hour_of_day'] / 24)

print('Temporal features added.')

Temporal features added.


/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/1121351322.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hour_features['hour_of_day'] = hour_features[timestamp_col].dt.hour
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/1121351322.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hour_features['day_of_week'] = hour_features[timestamp_col].dt.dayofweek
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/1121351322.py:4: PerformanceWarning: DataFrame is highly fragmented.  Th

In [6]:
# 5. Add rolling and lagged features
# Rolling features at hour level
rolling_windows = [6, 12, 24]

for metric in model_metrics:
    abs_resid_col = f'{metric}_abs_residual'
    robust_z_col = f'{metric}_robust_z'
    
    if abs_resid_col in hour_features.columns:
        for wh in rolling_windows:
            hour_features[f'{metric}_abs_resid_mean_{wh}h'] = (
                hour_features.groupby(sensor_col)[abs_resid_col]
                .rolling(window=wh, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )
    
    if robust_z_col in hour_features.columns:
        for wh in rolling_windows:
            hour_features[f'{metric}_robust_z_mean_{wh}h'] = (
                hour_features.groupby(sensor_col)[robust_z_col]
                .rolling(window=wh, min_periods=1)
                .mean()
                .reset_index(level=0, drop=True)
            )

# Lagged features
lag_hours = [1, 3, 6, 24]
for metric in model_metrics:
    abs_resid_col = f'{metric}_abs_residual'
    if abs_resid_col in hour_features.columns:
        for lh in lag_hours:
            hour_features[f'{metric}_abs_resid_lag_{lh}h'] = (
                hour_features.groupby(sensor_col)[abs_resid_col]
                .shift(lh)
            )

# Persistence counts (hours above threshold)
threshold = 2.0  # example threshold for robust z
for metric in model_metrics:
    robust_z_col = f'{metric}_robust_z'
    if robust_z_col in hour_features.columns:
        exceed = (hour_features[robust_z_col].abs() > threshold).astype(int)
        for wh in rolling_windows:
            hour_features[f'{metric}_persist_exceed_{wh}h'] = (
                exceed.groupby(hour_features[sensor_col])
                .rolling(window=wh, min_periods=1)
                .sum()
                .reset_index(level=0, drop=True)
            )

print('Rolling and lagged features added.')

/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/2644450057.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hour_features[f'{metric}_abs_resid_mean_{wh}h'] = (
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/2644450057.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hour_features[f'{metric}_robust_z_mean_{wh}h'] = (
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/2644450057.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling

Rolling and lagged features added.


/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/2644450057.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hour_features[f'{metric}_persist_exceed_{wh}h'] = (
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/2644450057.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  hour_features[f'{metric}_persist_exceed_{wh}h'] = (
/var/folders/nn/cc_nydqj2_n5tddmw2qsf4800000gn/T/ipykernel_82993/2644450057.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

In [8]:
# 6. Build 24-hour window feature table
# Group by sensor and 24h windows
hour_features['window_start'] = hour_features[timestamp_col].dt.floor('24h')

window_features = []
for (sensor, window), group in hour_features.groupby([sensor_col, 'window_start']):
    record = {
        sensor_col: sensor,
        'window_start': window,
        'window_end': window + pd.Timedelta(hours=24),
        'hours_with_data': group.dropna(subset=[f'{m}_raw' for m in model_metrics], how='all').shape[0]
    }
    
    for metric in model_metrics:
        abs_resid_col = f'{metric}_abs_residual'
        robust_z_col = f'{metric}_robust_z'
        
        if abs_resid_col in group.columns:
            valid_resids = group[abs_resid_col].dropna()
            if not valid_resids.empty:
                record.update({
                    f'{metric}_resid_max': valid_resids.max(),
                    f'{metric}_resid_mean': valid_resids.mean(),
                    f'{metric}_resid_median': valid_resids.median(),
                    f'{metric}_resid_rms': np.sqrt((valid_resids**2).mean())
                })
        
        if robust_z_col in group.columns:
            valid_z = group[robust_z_col].dropna()
            if not valid_z.empty:
                record.update({
                    f'{metric}_z_max': valid_z.abs().max(),
                    f'{metric}_z_mean': valid_z.abs().mean(),
                    f'{metric}_z_median': valid_z.abs().median()
                })
    
    # Coverage summaries
    record['mean_neighbour_coverage'] = group['neighbour_coverage_ratio'].mean()
    record['min_neighbour_coverage'] = group['neighbour_coverage_ratio'].min()
    record['fallback_hours'] = group['fallback_reason'].notna().sum()
    
    window_features.append(record)

window_df = pd.DataFrame(window_features)
print('Window features shape:', window_df.shape)
print('Window features head:', window_df.head())

Window features shape: (1056, 105)
Window features head:               ateccid              window_start                window_end  \
0  0123067D065E709F01 2025-08-15 00:00:00+00:00 2025-08-16 00:00:00+00:00   
1  0123067D065E709F01 2025-08-16 00:00:00+00:00 2025-08-17 00:00:00+00:00   
2  0123067D065E709F01 2025-08-17 00:00:00+00:00 2025-08-18 00:00:00+00:00   
3  0123067D065E709F01 2025-08-18 00:00:00+00:00 2025-08-19 00:00:00+00:00   
4  0123067D065E709F01 2025-08-19 00:00:00+00:00 2025-08-20 00:00:00+00:00   

   hours_with_data  mean_neighbour_coverage  min_neighbour_coverage  \
0               24                      0.0                     0.0   
1               24                      0.0                     0.0   
2               24                      0.0                     0.0   
3               24                      0.0                     0.0   
4               24                      0.0                     0.0   

   fallback_hours  tvoc_resid_max  tvoc_resid_mean  t

In [9]:
# 7. Data quality checks
# Check for duplicates
dup_cols = hour_features.columns[hour_features.columns.duplicated()].tolist()
if dup_cols:
    raise ValueError(f'Duplicate columns found: {dup_cols}')

dup_keys = hour_features.duplicated(subset=[sensor_col, timestamp_col]).sum()
if dup_keys > 0:
    raise ValueError(f'Duplicate sensor-time keys: {dup_keys}')

# All-null features
null_features = hour_features.columns[hour_features.isnull().all()].tolist()
if null_features:
    print(f'Warning: All-null features: {null_features}')

# Zero-variance features
var_features = []
for col in hour_features.select_dtypes(include=[np.number]).columns:
    if hour_features[col].var() == 0:
        var_features.append(col)
if var_features:
    print(f'Warning: Zero-variance features: {var_features}')

print('Data quality checks passed.')

Data quality checks passed.


In [10]:
# 8. Export canonical artefacts
# Sensor hour features
sensor_hour_features_path = outputs / 'sensor_hour_features.csv'
hour_features.to_csv(sensor_hour_features_path, index=False)
print('Saved sensor hour features:', sensor_hour_features_path)

# Sensor window features
sensor_window_features_path = outputs / 'sensor_window_features.csv'
window_df.to_csv(sensor_window_features_path, index=False)
print('Saved sensor window features:', sensor_window_features_path)

# Feature dictionary
feature_dict = []
for col in hour_features.columns:
    if col in [timestamp_col, sensor_col]:
        continue
    feature_dict.append({
        'feature_name': col,
        'feature_type': 'temporal' if 'hour' in col or 'day' in col else 'sensor',
        'metric': col.split('_')[0] if '_' in col else None,
        'description': f'Feature {col}'
    })

feature_dict_df = pd.DataFrame(feature_dict)
feature_dictionary_path = outputs / 'feature_dictionary.csv'
feature_dict_df.to_csv(feature_dictionary_path, index=False)
print('Saved feature dictionary:', feature_dictionary_path)

# Sensor neighbour metadata
sensor_meta = []
for sensor in sensor_list:
    neighs = neighbour_map.get(sensor, [])
    sensor_meta.append({
        sensor_col: sensor,
        'neighbour_count': len(neighs),
        'neighbours': ','.join(neighs)
    })

sensor_meta_df = pd.DataFrame(sensor_meta)
sensor_neighbour_metadata_path = outputs / 'sensor_neighbour_metadata.csv'
sensor_meta_df.to_csv(sensor_neighbour_metadata_path, index=False)
print('Saved sensor neighbour metadata:', sensor_neighbour_metadata_path)

print('All artefacts exported.')

Saved sensor hour features: ../data/sensor_hour_features.csv
Saved sensor window features: ../data/sensor_window_features.csv
Saved feature dictionary: ../data/feature_dictionary.csv
Saved sensor neighbour metadata: ../data/sensor_neighbour_metadata.csv
All artefacts exported.
